# 4. Essential LLM Use Cases

Practical live examples of summarization, sentiment analysis, NER, translation, multilingual processing, classification, keyword extraction and rewriting.

## 1. Reusable model

We use deterministic output for extraction/classification and slightly higher temperature only when creative rewriting is useful.

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

### 2. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
llm=ChatOpenAI(model=MODEL_NAME,temperature=0)
text="A deployment exhausted the connection pool at 09:10. Traffic was redirected, the release was rolled back, and service recovered at 10:05. No data was lost."
print(llm.invoke(f"Summarize in 3 bullets: impact, cause, resolution. Do not invent facts.\n{text}").content)

### 3. Define and validate structured output

This cell defines a Pydantic schema and configures structured output so model results have predictable fields and types.

**Expected result:** A validated Python object or dictionary is returned instead of unstructured text. Read the output before continuing to the next cell.

In [ ]:
from typing import Literal
from pydantic import BaseModel,Field
class Sentiment(BaseModel):
    label: Literal["Positive","Negative","Neutral"]
    confidence: float=Field(ge=0,le=1)
    reason: str
sentiment_model=llm.with_structured_output(Sentiment)
print(sentiment_model.invoke("Analyze: Support was polite, but the problem remains unresolved.").model_dump())

### 4. Define and validate structured output

This cell defines a Pydantic schema and configures structured output so model results have predictable fields and types.

**Expected result:** A validated Python object or dictionary is returned instead of unstructured text. Read the output before continuing to the next cell.

In [ ]:
class Entities(BaseModel):
    people:list[str]=[]; organizations:list[str]=[]; locations:list[str]=[]; dates:list[str]=[]
ner=llm.with_structured_output(Entities)
print(ner.invoke("Extract entities: Meera from ABC Technologies meets Ravi in Hyderabad on 20 September 2026.").model_dump())

### 5. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
for language in ["Hindi","Telugu","French"]:
    prompt=f"Translate to {language}. Preserve Cloud Portal and 10:00 PM exactly: Cloud Portal maintenance begins at 10:00 PM."
    print(language,":",llm.invoke(prompt).content)

### 6. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
mixed="Hello team, நாளை meeting है at 3 PM. Please confirm."
print(llm.invoke(f"Identify all languages, translate to English, and preserve the time exactly: {mixed}").content)

### 7. Define and validate structured output

This cell defines a Pydantic schema and configures structured output so model results have predictable fields and types.

**Expected result:** A validated Python object or dictionary is returned instead of unstructured text. Read the output before continuing to the next cell.

In [ ]:
class BusinessAnalysis(BaseModel):
    category: str; keywords:list[str]; urgency:Literal["Low","Medium","High"]; recommended_action:str
analyzer=llm.with_structured_output(BusinessAnalysis)
print(analyzer.invoke("Classify and extract keywords: Production checkout fails for all customers after deployment.").model_dump())

### 8. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
creative=ChatOpenAI(model=MODEL_NAME,temperature=0.5)
rough="send report today figures need check ask finance"
print(creative.invoke(f"Rewrite as a polite professional message without inventing details: {rough}").content)

## Validation and comparison

For summarization check faithfulness; for sentiment and classification check labels; for NER check exact spans; for translation use a fluent reviewer; for structured output validate schema and types.

## Practice

Add use cases for question answering, content generation, topic detection and document comparison. Create at least three test inputs for every use case and record failures.